<a href="https://colab.research.google.com/github/LeonimerMelo/Reinforcement-Learning/blob/Policy-Gradient/Soft_Actor_Critic_(SAC)_off_policy_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Soft Actor Critic (SAC)
O Soft Actor-Critic (SAC) é um algoritmo de Aprendizado por Reforço (Deep Reinforcement Learning) de ponta projetado para lidar com espaços de ação contínuos. Ele combina dois modelos (ator e crítico) usando o princípio de "máxima entropia", o que incentiva a exploração inteligente e garante alta estabilidade e eficiência na vida real.

O Soft Actor Critic (SAC) é um algoritmo que otimiza uma política estocástica de forma não-política, estabelecendo uma ponte entre a otimização de políticas estocásticas e abordagens do tipo DDPG. Não é um sucessor direto do TD3 (tendo sido publicado praticamente ao mesmo tempo), mas incorpora o truque do duplo-Q truncado e, devido à estocasticidade inerente à política no SAC, também acaba se beneficiando de algo como a suavização da política alvo.

Uma característica central do SAC é a regularização por entropia. A política é treinada para maximizar um equilíbrio entre o retorno esperado e a entropia , uma medida de aleatoriedade na política. Isso tem uma estreita relação com o equilíbrio entre exploração e explotação: aumentar a entropia resulta em mais exploração, o que pode acelerar o aprendizado posteriormente. Também pode impedir que a política convirja prematuramente para um ótimo local ruim.

###Características principais do SAC
- SAC is an off-policy algorithm.
- É comumente usada em ambientes com espaços de ação contínuos.
- Uma versão alternativa do SAC, que altera ligeiramente a regra de atualização da política, pode ser implementada para lidar com espaços de ação discretos.
- A implementação Spinning Up do SAC não suporta paralelização.

## Como o SAC funciona
O SAC baseia-se em quatro componentes principais trabalhando juntos:

* O Ator (Política Estocástica): Define qual ação tomar. Em vez de apenas cuspir um valor cego, ele gera uma distribuição de probabilidade (como uma curva normal ou sino) com média e desvio padrão. Isso permite que o agente explore naturalmente e varie suas ações.
* O Crítico (Funções Valor - Q-networks): Avalia a qualidade das ações escolhidas pelo Ator. O SAC geralmente usa duas redes de crítico separadas para mitigar o problema de superestimar recompensas (otimismo excessivo).
* Buffer de Repercussão (Replay Buffer): O agente grava suas experiências passadas (estado, ação, recompensa, próximo estado) em um histórico. Ele aprende amostrando mini-lotes aleatórios dessas experiências, economizando tempo e dados.
* Máxima Entropia (O Segredo do Sucesso): O objetivo do SAC não é só maximizar a recompensa (o "sucesso" da tarefa), mas também maximizar a entropia (uma medida de aleatoriedade). Em termos práticos, ele força o agente a encontrar a solução da tarefa, mas mantendo o máximo de aleatoriedade e diversidade possível nas suas decisões.

## Por que usar o SAC?

   1. Forte contra "becos sem saída": A alta entropia faz com que o agente não fique "preso" repetindo a mesma ação subótima, forçando-o a explorar continuamente maneiras diferentes de resolver o problema.
   2. Eficiência de dados: Como é um algoritmo off-policy, ele consegue aprender reutilizando dados antigos armazenados no buffer de experiência, o que o torna excelente até para robôs reais.
   3. Automático: A temperatura do parâmetro de entropia (frequentemente chamada de Alfa α) é ajustada automaticamente, o que poupa o trabalho de calibrar esse valor manualmente.

##Arquitetura do SAC

<center><img src='https://drive.google.com/uc?id=1lGM4-6rfzHPgywOb4ZwPdZCOyo2yfSsw' width=800></center>

##Diferenças entre SAC e A2C
Soft Actor-Critic (SAC) e Advantage Actor-Critic (A2C) são algoritmos fundamentais de aprendizado por reforço baseados na estrutura ator-crítico (actor-critic), mas diferem na coleta de dados, objetivos e ambientes recomendados. Enquanto o A2C atualiza a política de forma síncrona com experiências imediatas, o SAC otimiza um objetivo de máxima entropia usando dados históricos salvos em um histórico de jogo (replay buffer).
Aqui está a comparação detalhada entre os dois algoritmos.
### Visão Geral Comparativa

| Característica | Advantage Actor-Critic (A2C) | Soft Actor-Critic (SAC) |
|---|---|---|
| Política de Dados | On-policy (aprende estritamente com a política atual) | Off-policy (aprende com um buffer de experiências passadas) |
| Objetivo Principal | Maximizar o retorno esperado | Maximizar o retorno esperado + a entropia da política |
| Eficiência de Amostragem | Baixa (os dados não podem ser reutilizados) | Alta (as experiências são guardadas e reutilizadas) |
| Espaço de Ação | Preferido para Discreto (suporta Contínuo) | Nativo para Contínuo (adaptável para Discreto) |
| Método de Exploração | Política estocástica básica / penalidade de entropia | Regularização de Máxima Entropia integrada |
| Estabilidade | Altamente sensível a hiperparâmetros | Excepcionalmente robusto e estável |

------------------------------
### Diferenças Estruturais e Algorítmicas

####1. Aprendizado On-Policy vs. Off-Policy

* A2C é On-policy: Coleta um lote de experiências com a política atual, atualiza as redes e descarta esses dados imediatamente. Ele não aprende com dados gerados por versões antigas da política.
* SAC é Off-policy: Utiliza um replay buffer para armazenar transições (s, a, r, s'). O agente extrai amostras aleatórias desse buffer, reutilizando dados antigos várias vezes, o que aumenta muito a eficiência de amostragem.

####2. Objetivo Padrão vs. Máxima Entropia

* Objetivo do A2C: Focado estritamente em maximizar a recompensa acumulada. O Crítico estima uma função de valor de estado V(s) para calcular a Função de Vantagem A(s,a) = Q(s,a) - V(s), ajudando o Ator a entender quais ações foram melhores que a média.
* Objetivo do SAC: Opera na estrutura de Máxima Entropia. O objetivo equilibra a maximização das recompensas com a maximização da entropia (aleatoriedade) da política. A política aprende a resolver a tarefa agindo de forma tão ampla quanto possível, evitando convergência prematura para soluções ruins.

#### 3. Arquitetura da Rede

* Estrutura A2C: Geralmente possui duas saídas de rede (compartilhando camadas iniciais): uma para o Ator (distribuição da política) e outra para o Crítico (valor de estado V(s)).
* Estrutura SAC: Possui um sistema de redes mais complexo para evitar a superestimação de valores. Utiliza um Ator estocástico junto com Redes Críticas Gêmeas (Twin Critics Q), calculando o valor mínimo entre ambas para atualizar os alvos de forma estável.

------------------------------
### Quando Usar Cada Um?

* Escolha o A2C se: Você tiver simuladores altamente paralelos (o A2C coleta dados síncronos com vários trabalhadores), trabalhar com ações discretas ou precisar de uma linha de base simples e leve.
* Escolha o SAC se: Seu ambiente operar em domínios de ações contínuas complexas (como robótica, direção autônoma ou controle físico), onde interagir com o ambiente é demorado ou custoso.





##Referências
[1] [https://spinningup.openai.com](https://translate.google.com/translate?u=https://spinningup.openai.com/en/latest/algorithms/sac.html&hl=pt&sl=en&tl=pt&client=sge)

[2] [https://www.youtube.com](https://www.youtube.com/watch?v=ApG0lWv6gGc)

[3] [https://www.geeksforgeeks.org](https://www.geeksforgeeks.org/deep-learning/soft-actor-critic-reinforcement-learning-algorithm/)

[4] [https://rlinf.readthedocs.io](https://rlinf.readthedocs.io/en/latest/rst_source/tutorials/rlalg/sac.html)

[5] [https://www.youtube.com](https://www.youtube.com/watch?v=ioidsRlf79o)

[6] [https://pt.linkedin.com](https://pt.linkedin.com/advice/3/what-some-practical-applications-soft-actor-critic?lang=pt)

[7] [https://www.mathworks.com](https://translate.google.com/translate?u=https://www.mathworks.com/help/reinforcement-learning/ug/soft-actor-critic-agents.html&hl=pt&sl=en&tl=pt&client=sge)

[8] [https://bair.berkeley.edu](https://bair.berkeley.edu/blog/2018/12/14/sac/)

[9] [https://arxiv.org](https://arxiv.org/abs/1812.05905)